In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import re
import matplotlib.pyplot as plt
import copy
import math
import h5py

In [ ]:
# efel_dirs = ['allen_full_11_09_21_488683423',
#              'compare_bbp_full_06_19_23_485574832',
#             #'compare_bbp_full_06_21_23_480351780',]
#              'compare_bbp_full_06_22_23_314822529',
#             'compare_bbp_full_06_22_23_468120757',
#             "compare_bbp_full_06_30_23_314831019",
#             'compare_bbp_full_07_07_23_484559000']

cell_ids = [ '484559000',
            '480351780',
            '314822529',
            '468120757',
            "314831019",
            '485574832']

# cell_ids = ['484559000', '314831019']

models = ['allen', 'M1_TTPC_NA_HH', 'compare_bbp', 'cell']


In [ ]:

def get_cell_num(dirname):
    cell_num = re.findall(r'\d+', dirname)[-1]
    return cell_num

def merge_dicts(data1, data2={}):
    res = copy.deepcopy(data1)
    for key in data2.keys():
        if type(data2[key]) == dict and type(data1[key])  == dict:
            res[key] = merge_dicts(data2[key],data1[key])
        else:
            res[key] = data2[key]
    return res


def find_volts_folder(base):
    volts = os.path.join(base, 'genetic_alg', 'compare_responses')
    files = os.listdir(volts)
    files = [os.path.join(volts, f) for f in folders if 'csv' in f] # add path to each file
    return files

def volts_to_dict(volts_path):
    res = {}
    for file in volts_path:
        num = os.path.basename(file).replace('.csv', '')
        res[num] = np.genfromtxt(file, delimiter=',')
    return res

def find_and_parse_sota_nwb(base, model):
    sota = os.path.join(base, 'genetic_alg')
    hdf = [elem for elem in os.listdir(sota) if 'sota' in elem]
    if len(hdf) == 0:
        return {}
        # import pdb; pdb.set_trace()
    data = h5py.File(os.path.join(sota, hdf[0]), 'r')
    res = {}
    for key in data.keys():
        if 'allen_model_response' in key and model == 'allen':
            new_key = key.split('_')[0]
            res[new_key] = data[key][:]
        elif 'cell_response' in key and model == 'cell':
            new_key = key.split('_')[0]
            res[new_key] = data[key][:]
        elif  'compare_model_response' in key and \
        (model == 'compare_bbp' or model == 'M1_TTPC_NA_HH'):
            new_key = key.split('_')[0]
            res[new_key] = data[key][:]
    data.close()
    return res


In [ ]:
all_cell_efel_data = {}
df_rows = []
for cell_id in cell_ids:
    data = {}
    all_cell_efel_data[cell_id] = {}
    for model_idx, model in enumerate(models):
        if model not in ['cell', 'allen']:
            candidates = [folder for folder in os.listdir() if model in folder and cell_id in folder]
            if not len(candidates): 
                print('no ', model, cell_id)
                continue
            curr_e_dir = candidates[-1]
            all_cell_efel_data[cell_id][model] = find_and_parse_sota_nwb(curr_e_dir, model)
        else:
            candidates = [folder for folder in os.listdir() if 'compare_bbp' in folder and cell_id in folder]
            if not len(candidates): 
                print('no ', model, cell_id)
                continue
            curr_e_dir = candidates[-1]
            all_cell_efel_data[cell_id][model] = find_and_parse_sota_nwb(curr_e_dir, model)

            
    # df_rows += df_rows_from_data(data, cell_num)


In [ ]:
all_cell_efel_data.keys()

In [ ]:
all_cell_efel_data['484559000']['M1_TTPC_NA_HH'].keys()

In [ ]:
plt.plot(all_cell_efel_data['484559000']['M1_TTPC_NA_HH']['55'][::20])

In [ ]:
with open('volt_data.pkl','wb') as f:
    pickle.dump(all_cell_efel_data, f)

In [ ]:
all_cell_efel_data.keys()

In [ ]:
all_cell_efel_data['484559000']['allen']['43'] \
== all_cell_efel_data['484559000']['cell']['43']

In [ ]:
plt.plot(all_cell_efel_data['484559000']['cell']['43'])
plt.plot(all_cell_efel_data['484559000']['allen']['43'])

In [ ]:
def rmse(x1,x2):
    return np.sqrt(np.mean((x1-x2)**2))

In [ ]:
rmse_rows = []
for cell_id in all_cell_efel_data.keys():
    for model in all_cell_efel_data[cell_id].keys():
        if model == 'cell': continue
        for stim in all_cell_efel_data[cell_id][model]:
            curr_rmse = rmse(all_cell_efel_data[cell_id][model][stim], all_cell_efel_data[cell_id]['cell'][stim])
            rmse_rows.append({'cell_id': cell_id, 'model': model, 'stim': stim, 'rmse' : curr_rmse})

In [ ]:
df = pd.DataFrame(rmse_rows)
df

In [ ]:
df.groupby(['cell_id','model']).agg(['mean','sem']).drop('stim',axis=1)

In [ ]:
cond = (df['cell_id'] == '314831019') &  ((df['model'] == 'M1_TTPC_NA_HH') | (df['model'] == 'compare_bbp'))
pd.set_option('display.max_rows', 500)

df.loc[cond].groupby(['model','stim']).agg(np.mean)